In [ ]:
import os
import glob
import ast
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import torch
from libpysal.weights import KNN
from esda.moran import Moran
from scipy.optimize import minimize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# --- Circular Statistics Helpers ---

def circular_mean(angles):
    """Computes the directional/circular mean of an array of angles (in radians)."""
    return np.arctan2(np.sum(np.sin(angles)), np.sum(np.cos(angles)))


def circular_correlation(alpha, beta):
    """Computes the Jammalamadaka–Sarma circular correlation coefficient (radians)."""
    mu_alpha = circular_mean(alpha)
    mu_beta = circular_mean(beta)

    numerator = np.sum(np.sin(alpha - mu_alpha) * np.sin(beta - mu_beta))
    denominator = np.sqrt(
        np.sum(np.sin(alpha - mu_alpha) ** 2)
        * np.sum(np.sin(beta - mu_beta) ** 2)
    )

    return numerator / denominator if denominator != 0 else 0.0


def mean_resultant_length(phi, theta):
    """
    Mean resultant length of angular differences delta = theta - phi.
    R in [0, 1].  R ≈ 1 means the labels align perfectly with spatial angles
    around the chosen center (up to a global rotation).
    """
    delta = theta - phi
    C = np.mean(np.cos(delta))
    S = np.mean(np.sin(delta))
    return np.sqrt(C**2 + S**2)


def neg_abs_circ_corr(center, points, labels):
    """
    Objective for the optimizer: negative |circular correlation|
    between spatial angles around 'center' and the labels.
    """
    dx = points[:, 0] - center[0]
    dy = points[:, 1] - center[1]
    # Guard against the optimizer landing exactly on a data point
    if np.any((np.abs(dx) < 1e-12) & (np.abs(dy) < 1e-12)):
        return 0.0
    phi = np.arctan2(dy, dx)
    corr = circular_correlation(phi, labels)
    if np.isnan(corr):
        return 0.0
    return -abs(corr)


def find_optimal_center(points, labels, init_guesses=None, method="Nelder-Mead"):
    """
    Find the center (a, b) that maximizes |circular_correlation(phi, labels)|.
    """
    if init_guesses is None:
        mean_c = np.mean(points, axis=0)
        bbox_c = (np.min(points, axis=0) + np.max(points, axis=0)) / 2.0
        init_guesses = [mean_c, bbox_c]

    best_center = None
    best_corr = -1.0
    best_mrl = 0.0

    for guess in init_guesses:
        result = minimize(
            neg_abs_circ_corr,
            guess,
            args=(points, labels),
            method=method,
            options={"xatol": 1e-7, "fatol": 1e-7, "maxiter": 500},
        )
        corr = -result.fun
        if corr > best_corr:
            best_corr = corr
            best_center = result.x

    # Compute MRL at the winning center
    phi_opt = np.arctan2(points[:, 1] - best_center[1], points[:, 0] - best_center[0])
    best_mrl = mean_resultant_length(phi_opt, labels)

    return best_center, best_corr, best_mrl


In [ ]:
def analyze_spatial_autocorrelation(
    var_title,
    geom_csv_path,
    meta_idx,
    latents_path,
    device="cpu",
    circular=False,
    time=False,
):
    """
    Analyze spatial autocorrelation for a given transformation and latent space.
    
    Always plots the spatial scatter colored by the target variable and computes
    Moran's I. For circular variables, also computes circular correlation and
    marks the optimal center.
    
    Parameters
    ----------
    var_title : str
        Human-readable variable name (e.g. "Altitude", "Latitude").
    geom_csv_path : str
        Path to the geometry CSV (rotation_matrix, translation_vec, lambda_vec).
    meta_idx : int
        Column index in meta tensor to use as labels.
    latents_path : str
        Path to the latents .pt file.
    device : str
        'cpu' or 'cuda'.
    circular : bool
        If True, treat variable as cyclic (twilight colormap + circular stats).
    time : bool
        If True, apply time-specific latitude/altitude filtering.
    """
    # ---------- Load Geometry ----------
    df_geom = pd.read_csv(geom_csv_path)
    columns_to_convert = ["rotation_matrix", "translation_vec", "lambda_vec"]
    for col in columns_to_convert:
        df_geom[col] = df_geom[col].apply(ast.literal_eval)

    # Each transformation file should contain a single row for the selected seed
    geom_row = df_geom.iloc[0]
    R_m = torch.tensor(geom_row["rotation_matrix"], device=device, dtype=torch.float32)
    t_m = torch.tensor(geom_row["translation_vec"], device=device, dtype=torch.float32)
    l_m = torch.tensor(geom_row["lambda_vec"], device=device, dtype=torch.float32)

    # ---------- Load Latents ----------
    data = torch.load(latents_path, map_location=device)

    if time:
        current_meta = data["meta"]
        lats = current_meta[:, 0]
        alts = current_meta[:, 3]
        mask = (lats > 15) & (lats < 66) & (alts < 4000)
        X = data["latents"][mask]
        labels = current_meta[:, -1][mask].cpu().numpy().flatten()
    else:
        X = data["latents"]
        labels = data["meta"][:, meta_idx].cpu().numpy().flatten()
        if circular:
            labels = labels * np.pi / 180

    # ---------- Apply Transformation ----------
    with torch.no_grad():
        X_centered_m = X.to(device) + t_m
        X_rotated_m = torch.matmul(X_centered_m, R_m)
        top_2_m = torch.argsort(l_m.abs(), descending=True)[:2]
        Ls_model = X_rotated_m[:, top_2_m].cpu().numpy()

    # ---------- Moran's I ----------
    gdf_m = gpd.GeoDataFrame(
        {"label": labels},
        geometry=gpd.points_from_xy(Ls_model[:, 0], Ls_model[:, 1]),
    )
    w_m = KNN.from_dataframe(gdf_m, k=8)
    w_m.transform = "R"
    mi = Moran(gdf_m["label"], w_m)

    # ---------- Plot ----------
    cmap_choice = "twilight" if circular else "viridis"

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        Ls_model[:, 0],
        Ls_model[:, 1],
        c=labels,
        cmap=cmap_choice,
        s=5,
        alpha=0.5,
    )
    plt.colorbar(scatter, label=var_title)

    if circular:
        center_m_opt, corr_m_opt, mrl_m_opt = find_optimal_center(Ls_model, labels)
        center_m_mean = np.mean(Ls_model, axis=0)
        center_m_bbox = (np.min(Ls_model, axis=0) + np.max(Ls_model, axis=0)) / 2.0

        plt.scatter(
            center_m_opt[0],
            center_m_opt[1],
            color="lime",
            marker="*",
            s=250,
            edgecolor="black",
            label=f"Optimal Center (r={round(corr_m_opt, 2)}, R={round(mrl_m_opt, 2)})",
            zorder=4,
        )
        plt.scatter(
            center_m_mean[0],
            center_m_mean[1],
            color="red",
            marker="X",
            s=150,
            edgecolor="black",
            label="Mean Center",
            zorder=3,
        )
        plt.scatter(
            center_m_bbox[0],
            center_m_bbox[1],
            color="cyan",
            marker="o",
            s=100,
            facecolors="none",
            edgecolors="cyan",
            linewidths=2,
            label="BBox Center",
            zorder=2,
        )
        plt.legend(loc="best")

    plt.title(f"{var_title} — Moran's I = {mi.I:.4f} (p = {mi.p_sim:.4f})")
    plt.xlabel("Latent Dim 1")
    plt.ylabel("Latent Dim 2")
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    # ---------- Print Summary ----------
    print(f"\n--- {var_title} Spatial Autocorrelation ---")
    print(f"Moran's I:        {mi.I:.6f}")
    print(f"Moran's I p-val:  {mi.p_sim:.6f}")
    print(f"Moran's I z-score:{mi.z_sim:.6f}")

    if circular:
        print(f"Circular Corr:    {corr_m_opt:.6f}")
        print(f"Mean Res. Length: {mrl_m_opt:.6f}")

    return mi.I, mi.p_sim, mi.z_sim


In [ ]:
# ============================================================
# CONFIGURATION — adjust these values to inspect different cases
# ============================================================

# Which iterations to consider (e.g. [2, 4, 6, 8, 10, 12])
TARGET_ITERATIONS = [2, 4, 6, 8, 10, 12]

# Metric to rank by: "moran_I" or "circ_corr"
# NOTE: circ_corr only exists / is meaningful for circular variables (T, Long).
#       For H and L, moran_I is the sensible choice.
METRIC = "moran_I"

# Order: "lowest" or "highest"
ORDER = "lowest"

# Rank: 1 = lowest/highest, 2 = 2nd lowest/highest, etc.
RANK = 1

# Directories (relative to notebooks/)
RESULTS_DIR = "../results"
TRANSFORMATIONS_DIR = "../transformations"
DATA_DIR = "../data"

# Variable definitions
VARIABLES = {
    "H": {
        "title": "Altitude",
        "meta_idx": 3,
        "circular": False,
        "time": False,
    },
    "L": {
        "title": "Latitude",
        "meta_idx": 0,
        "circular": False,
        "time": False,
    },
    "Long": {
        "title": "Longitude",
        "meta_idx": 1,
        "circular": True,
        "time": False,
    },
    "T": {
        "title": "Season",
        "meta_idx": -1,
        "circular": True,
        "time": True,
    },
}


def find_transformation_file(transformations_dir, seed):
    """Find the transformation CSV matching a given seed."""
    # Direct match: {seed}.csv
    direct = os.path.join(transformations_dir, f"{seed}.csv")
    if os.path.exists(direct):
        return direct

    # Pattern: {seed}_geometry_results*.csv
    pattern = os.path.join(transformations_dir, f"{seed}_geometry_results*.csv")
    matches = glob.glob(pattern)
    if matches:
        return matches[0]

    # Fallback: any CSV containing the seed name
    pattern = os.path.join(transformations_dir, f"*{seed}*.csv")
    matches = glob.glob(pattern)
    if matches:
        return matches[0]

    return None


# ============================================================
# SCAN & INSPECT
# ============================================================

for var_code, var_config in VARIABLES.items():
    results_file = os.path.join(RESULTS_DIR, f"{var_code}_results.csv")
    if not os.path.exists(results_file):
        print(f"\n⚠️  Results file not found: {results_file}")
        continue

    df = pd.read_csv(results_file)

    # Filter to target iterations
    df_filtered = df[df["iteration"].isin(TARGET_ITERATIONS)]

    if df_filtered.empty:
        print(f"\n⚠️  No data for {var_code} with iterations {TARGET_ITERATIONS}")
        continue

    # Check if metric column exists
    if METRIC not in df_filtered.columns:
        print(f"\n⚠️  Metric '{METRIC}' not found in {var_code}_results.csv. Skipping.")
        continue

    # Sort by metric
    ascending = (ORDER == "lowest")
    df_sorted = df_filtered.sort_values(by=METRIC, ascending=ascending)

    # Pick the RANK-th row (1-indexed)
    if len(df_sorted) < RANK:
        print(f"\n⚠️  Not enough results for {var_code} (found {len(df_sorted)}, need {RANK})")
        continue

    selected = df_sorted.iloc[RANK - 1]
    seed = selected["seed"]
    iteration = int(selected["iteration"])

    print("\n" + "=" * 70)
    print(f"  {var_config['title']} — {RANK}{'th' if RANK > 3 else {1:'st',2:'nd',3:'rd'}.get(RANK,'th')} {ORDER} {METRIC}")
    print("=" * 70)
    print(f"  Seed:      {seed}")
    print(f"  Iteration: {iteration}")
    print(f"  {METRIC}:  {selected[METRIC]:.6f}")
    print("=" * 70)

    # Find transformation file
    trans_file = find_transformation_file(TRANSFORMATIONS_DIR, seed)
    if trans_file is None:
        print(f"  ❌ Transformation file not found for seed: {seed}")
        continue

    print(f"  📁 Transformation file: {trans_file}")

    # Construct latents path
    latents_path = os.path.join(DATA_DIR, f"latents_{iteration}_92160_for_intervention.pt")
    if not os.path.exists(latents_path):
        print(f"  ❌ Latents file not found: {latents_path}")
        continue

    print(f"  📁 Latents file:        {latents_path}")

    # Run analysis
    analyze_spatial_autocorrelation(
        var_title=var_config["title"],
        geom_csv_path=trans_file,
        meta_idx=var_config["meta_idx"],
        latents_path=latents_path,
        device=device,
        circular=var_config["circular"],
        time=var_config["time"],
    )


In [ ]:
# ============================================================
# OPTIONAL: Manual inspection of a specific seed/iteration
# ============================================================
# Uncomment and modify the block below to inspect a specific case
# without running the full scanning cell.

# analyze_spatial_autocorrelation(
#     var_title="Altitude",
#     geom_csv_path="../transformations/H_geom_seed42.csv",
#     meta_idx=3,
#     latents_path="../data/latents_10_92160_for_intervention.pt",
#     device=device,
#     circular=False,
#     time=False,
# )
